# Introductie Computer Vision met MNIST & CNNs
Deze notebook laat zien hoe afbeeldingen worden omgezet naar tensors en hoe een CNN werkt.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

## 2. MNIST dataset laden


De MNIST dataset levert oorspronkelijk afbeeldingen als:

- **Type:** PIL Image
- **Pixelwaarden:** 0–255

### Waarom is schalen belangrijk?

Als we **0–255** gebruiken:
- Grote getallen
- Instabiele gradients
- Trager leren

Als we **0–1** gebruiken (met `ToTensor()`):
- Betere werking van activatiefuncties (zoals ReLU)
- Snellere en stabielere training

👉 Daarom gebruiken we `transforms.ToTensor()` — deze zet niet alleen om naar een tensor, maar schaalt ook de waarden.


In [ ]:
# Zonder transform
raw_dataset = datasets.MNIST(root='./data', train=True, download=True)

img, _ = raw_dataset[0]
print("Zonder ToTensor:", type(img))

# Met transform
tensor_dataset = datasets.MNIST(root='./data', train=True, transform=transforms.ToTensor())

img_tensor, _ = tensor_dataset[0]
print("Met ToTensor:", type(img_tensor))
print("Shape:", img_tensor.shape)
print("Min/Max:", img_tensor.min(), img_tensor.max())

### Uitleg
- Zonder `ToTensor()` krijg je een **PIL Image**
- Met `ToTensor()` krijg je een **PyTorch tensor**
- Pixelwaarden worden geschaald van **0–255 naar 0–1**
- Er wordt een extra dimensie toegevoegd voor het kanaal: `[1, 28, 28]`

## 3. Bekijk wat voorbeelden

In [ ]:
batch_size_train = 6000
data_loader = torch.utils.data.DataLoader(tensor_dataset,batch_size=batch_size_train, shuffle=True)

examples = enumerate(data_loader)
batch_idx, (example_data, example_targets) = next(examples)
import matplotlib.pyplot as plotter
print(f'data dim:{example_data.shape}')
print(f'range:{example_data.min()} - {example_data.max()}')
input_dim = example_data.shape[1:]
fig = plotter.figure()
for i in range(6):
    plotter.subplot(2,3,i+1)
    plotter.tight_layout()
    plotter.imshow(example_data[i][0], cmap='gray', interpolation='none')
    plotter.title(f"class: {example_targets[i]}")
    plotter.xticks([])
    plotter.yticks([])

In [ ]:
import numpy as np

image, label = tensor_dataset[1]

print("Label:", label)
print("Shape:", image.shape)

plt.imshow(image.squeeze(), cmap='gray')
plt.title(f"Label: {label}")
plt.show()

np.set_printoptions(precision=2, suppress=True)
print(image.squeeze().numpy())

In [ ]:
img = image.squeeze()

plt.figure(figsize=(6,6))
plt.imshow(img, cmap='gray')
plt.title(f"Label: {label}")

# pixelwaarden tonen
for i in range(28):
    for j in range(28):
        value = img[i, j].item()
        if value > 0.2:  # alleen zichtbare pixels
            plt.text(j, i, f"{value:.1f}", ha='center', va='center', color='red', fontsize=6)

plt.axis('off')
plt.show()

Een afbeelding lijkt voor ons gewoon een plaatje, maar voor een computer is het eigenlijk een verzameling getallen. In het geval van MNIST is elke afbeelding een raster van 28 bij 28 pixels. Elke pixel heeft een waarde die aangeeft hoe licht of donker die pixel is.

Na het gebruik van ToTensor() liggen deze waarden tussen 0 en 1:

0 betekent zwart
1 betekent wit
waarden ertussen zijn grijstinten

Als we de afbeelding tonen met plt.imshow(), zien wij een cijfer. Maar als we de pixelwaarden printen, zien we dat het eigenlijk een matrix van getallen is.

Door ook de getallen in de afbeelding te tekenen, kun je precies zien welke waarde bij elke pixel hoort. Dit helpt om te begrijpen dat een neuraal netwerk geen “plaatjes” ziet zoals wij, maar gewoon rekent met getallen.

Je kunt het zo zien: wat wij herkennen als een handgeschreven cijfer, ziet het model als een verzameling van 784 getallen (28 × 28), waarop het berekeningen uitvoert om te bepalen welk cijfer het is.